Ячейка 1: Инициализация проекта и импорт библиотек

In [1]:
import os
import json
import re
import pandas as pd
from pathlib import Path
from datetime import datetime

# Определяем пути к данным
RAW_DATA_DIR = Path("../data/raw_things/")
OUTPUT_DIR = Path("../data/output/")

# Создаем папку для выгрузки, если её еще нет
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Форматируем текущую дату и время (ГодМесяцДень_ЧасыМинутыСекунды)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = OUTPUT_DIR / f"classmods_result_{timestamp}.csv"

print(f"Библиотеки импортированы. Рабочие директории настроены.")
print(f"Файл при экспорте будет сохранен как: {output_file.name}")

Библиотеки импортированы. Рабочие директории настроены.
Файл при экспорте будет сохранен как: classmods_result_20260728_170024.csv


Ячейка 2: Загрузка баз данных и построение глобального словаря перевода

In [2]:
# Загружаем основные базы данных
with open(RAW_DATA_DIR / "NexusConfigStoreInventory.json", "r", encoding="utf-8") as f:
    inventory_data = json.load(f)
with open(RAW_DATA_DIR / "NexusConfigStoreInventoryNamePart.json", "r", encoding="utf-8") as f:
    name_parts_data = json.load(f)
with open(RAW_DATA_DIR / "NexusConfigStore_StatDisplay.json", "r", encoding="utf-8") as f:
    stat_display_data = json.load(f)

# Загружаем также базу данных прошивок
firmware_path = RAW_DATA_DIR / "NexusConfigStoreFirmware.json"
firmware_data = {}
if firmware_path.exists():
    with open(firmware_path, "r", encoding="utf-8") as f:
        firmware_data = json.load(f)

def clean_bbcode(text):
    if not text: return ""
    cleaned = re.sub(r"\[.*?\]", "", text)
    if " - " in cleaned:
        cleaned = cleaned.split(" - ")[0]
    return cleaned.strip()

# Строим глобальную ВЛОЖЕННУЮ карту перевода для исключения перекрестного смешивания имен
part_translation_map = {}

# 1. Автоматический перевод через NamePart и StatDisplay (по всем категориям инвентаря)
for category, cat_val in inventory_data.items():
    if not isinstance(cat_val, dict): continue
    parts_dict = cat_val.get("parts", {})
    if not isinstance(parts_dict, dict): continue
    
    cat_lower = category.lower()
    part_translation_map[cat_lower] = {} # Инициализируем вложенный словарь для этой категории
        
    for part_id, part_val in parts_dict.items():
        if not isinstance(part_val, dict): continue
        part_name = part_val.get("name", "")
        
        # Разрешаем к переводу любые детали, включая корпуса и прошивки
        if not isinstance(part_name, str) or not any(part_name.startswith(p) for p in ["part_", "passive_", "body_", "leg_"]):
            continue
        
        fields = part_val.get("fields", {})
        if not isinstance(fields, dict): continue
            
        aspects = fields.get("Aspects", [])
        for aspect in aspects:
            if not isinstance(aspect, dict): continue
            
            # Вариант А: Ищем в аспектах именования InventoryNamingAspect
            if "InventoryNamingAspect" in aspect.get("structtype", ""):
                title_list = aspect.get("TitlePartList", []) or []
                prefix_list = aspect.get("PrefixPartList", []) or []
                suffix_list = aspect.get("SuffixPartList", []) or []
                for naming_def in title_list + prefix_list + suffix_list:
                    if isinstance(naming_def, str):
                        match = re.search(r"'(np_.*?)'", naming_def)
                        if match:
                            np_key = match.group(1)
                            if np_key in name_parts_data:
                                real_name = name_parts_data[np_key].get("fields", {}).get("PartName", "")
                                if real_name:
                                    part_translation_map[cat_lower][part_name.lower()] = real_name
                                    part_translation_map[cat_lower][part_name.replace("part_", "").lower()] = real_name
                                    
            # Вариант Б: Ищем в аспектах интерфейса UIStatAspect
            elif "UIStatAspect" in aspect.get("structtype", ""):
                ui_stats = aspect.get("UIStatsToInclude", []) or []
                for stat_def in ui_stats:
                    if isinstance(stat_def, str):
                        match = re.search(r"'(uistat_.*?)'", stat_def)
                        if match:
                            lp_key = match.group(1)
                            if lp_key in stat_display_data:
                                format_text = stat_display_data[lp_key].get("fields", {}).get("StatValue", {}).get("FormatText", "")
                                if format_text:
                                    real_lp_name = clean_bbcode(format_text)
                                    if real_lp_name:
                                        real_lp_name = re.sub(r"\\+?\\{.*?\\}%?\\s*", "", real_lp_name).strip()
                                        if real_lp_name:
                                            part_translation_map[cat_lower][part_name.lower()] = real_lp_name
                                            part_translation_map[cat_lower][part_name.replace("part_", "").lower()] = real_lp_name

# 2. Добавляем перевод прошивок напрямую из FirmwareDef в глобальный родительский пул ClassMod
global_com_key = "234 | classmod"
if global_com_key in part_translation_map:
    for fw_key, fw_val in firmware_data.items():
        if isinstance(fw_val, dict):
            fw_name = fw_val.get("fields", {}).get("Name", "")
            if fw_name:
                fw_key_clean = fw_key.lower().replace("fw_", "")
                part_translation_map[global_com_key][fw_key.lower()] = fw_name
                part_translation_map[global_com_key][f"part_firmware_{fw_key_clean}"] = fw_name
                part_translation_map[global_com_key][f"part_firmware_{fw_key.lower()}"] = fw_name

print(f"Базы данных загружены! Построен вложенный словарь перевода деталей по категориям: {len(part_translation_map)} категорий.")

Базы данных загружены! Построен вложенный словарь перевода деталей по категориям: 458 категорий.


Ячейка 3: Сбор Epic, Legendary и Pearlescent компонентов

In [3]:
legendary_coms = []

# Фильтруем только категории, относящиеся к классмодам
character_com_categories = [
    "254 | classmod_dark_siren", 
    "255 | classmod_paladin", 
    "256 | classmod_exo_soldier", 
    "259 | classmod_gravitar", 
    "404 | classmod_robodealer"
]

for category in character_com_categories:
    cat_val = inventory_data.get(category, {})
    if not isinstance(cat_val, dict): continue
    parts_dict = cat_val.get("parts", {})
    if not isinstance(parts_dict, dict): continue
        
    for part_id, part_val in parts_dict.items():
        if not isinstance(part_val, dict): continue
        part_path = part_val.get("path", "")
        if not isinstance(part_path, str): continue
            
        is_epic = "comp_04" in part_path
        is_legendary = "comp_05" in part_path
        is_pearlescent = "comp_06" in part_path
        
        if is_epic or is_legendary or is_pearlescent:
            fields = part_val.get("fields", {})
            if not isinstance(fields, dict): continue
                
            is_exclude = fields.get("bExcludeFromGlobalPool", False)
            world_drop_flag = not is_exclude
            
            selection_rules = fields.get("PartTypeSelectionRules", {})
            if not isinstance(selection_rules, dict):
                selection_rules = {}
                
            rarity_str = "Pearlescent" if is_pearlescent else ("Legendary" if is_legendary else "Epic")
            
            legendary_coms.append({
                "Item_Code": part_path,
                "Internal_Category": category,
                "Rarity": rarity_str,
                "World_Drop": world_drop_flag,
                "Manufacturer": "Unknown",
                "Display_Name": "Unknown",
                "Drop_Source": "Unknown",
                "Selection_Rules": selection_rules
            })

df = pd.DataFrame(legendary_coms)
df = df.drop_duplicates(subset=["Item_Code"]).reset_index(drop=True)

print(f"Сбор классмодов завершен. Найдено уникальных высокоуровневых предметов: {len(df)}")
df.head()

Сбор классмодов завершен. Найдено уникальных высокоуровневых предметов: 55


,Item_Code,Internal_Category,Rarity,World_Drop,Manufacturer,Display_Name,Drop_Source,Selection_Rules
0,classmod_dark_siren.comp_05_legendary_06,254 | classmod_dark_siren,Legendary,True,Unknown,Unknown,Unknown,"{'class_mod_body': {'PartCount': {'min': 1, 'M..."
1,classmod_dark_siren.comp_05_legendary_05,254 | classmod_dark_siren,Legendary,True,Unknown,Unknown,Unknown,"{'class_mod_body': {'PartCount': {'min': 1, 'M..."
2,classmod_dark_siren.comp_05_legendary_04,254 | classmod_dark_siren,Legendary,True,Unknown,Unknown,Unknown,"{'class_mod_body': {'PartCount': {'min': 1, 'M..."
3,classmod_dark_siren.comp_05_legendary_03,254 | classmod_dark_siren,Legendary,True,Unknown,Unknown,Unknown,"{'class_mod_body': {'PartCount': {'min': 1, 'M..."
4,classmod_dark_siren.comp_05_legendary_02,254 | classmod_dark_siren,Legendary,True,Unknown,Unknown,Unknown,"{'class_mod_body': {'PartCount': {'min': 1, 'M..."


Ячейка 4: Привязка персонажей (Vault Hunters)

In [4]:
# Карта точного сопоставления категорий классмодов с персонажами Borderlands 4
character_com_mapping = {
    "classmod_dark_siren": "Vex (Siren)",
    "classmod_paladin": "Amon (Paladin/Forgeknight)",
    "classmod_exo_soldier": "Rafa (ExoSoldier)",
    "classmod_gravitar": "Harlowe (Gravitar)",
    "classmod_robodealer": "C4SH (Robodealer)"
}

def determine_com_character(row):
    category = row["Internal_Category"]
    cat_prefix = category.split(" | ")[-1].strip().lower() if " | " in category else category.strip().lower()
    return character_com_mapping.get(cat_prefix, "Unknown")

df["Character"] = df.apply(determine_com_character, axis=1)
print("Привязка классмодов к Искателям Хранилища успешно выполнена!")
df[["Item_Code", "Character"]].head()

Привязка классмодов к Искателям Хранилища успешно выполнена!


,Item_Code,Character
0,classmod_dark_siren.comp_05_legendary_06,Vex (Siren)
1,classmod_dark_siren.comp_05_legendary_05,Vex (Siren)
2,classmod_dark_siren.comp_05_legendary_04,Vex (Siren)
3,classmod_dark_siren.comp_05_legendary_03,Vex (Siren)
4,classmod_dark_siren.comp_05_legendary_02,Vex (Siren)


Ячейка 5: Расшифровка названий классмодов

In [5]:
def resolve_display_name(item_code, category, name_parts):
    if not isinstance(item_code, str):
        return "Unknown"
        
    cat_clean = category.split(" | ")[-1].strip().lower() if " | " in category else category.strip().lower()
    
    # Карта сокращений для точного поиска ключей в NamePart
    prefix = {
        "classmod_dark_siren": "ds",
        "classmod_paladin": "pal",
        "classmod_exo_soldier": "exo",
        "classmod_gravitar": "grav",
        "classmod_robodealer": "robo"
    }.get(cat_clean, "")
    
    if not prefix:
        return "Unknown"
        
    # Извлекаем суффикс названия из кода предмета
    match = re.search(r"comp_0._(?:legendary_|epic_)?(.*)", item_code, re.IGNORECASE)
    if match:
        suffix = match.group(1).lower().replace("_", "")
        
        is_legendary = "comp_05" in item_code
        is_pearl = "comp_06" in item_code
        is_epic = "comp_04" in item_code
        
        # Строим точные системные маски ключей разработчиков
        target_keys = []
        if is_legendary:
            target_keys.append(f"np_cm_{prefix}_leg_{suffix}")
        elif is_pearl:
            target_keys.append(f"np_cm_{prefix}_pearl_{suffix}")
            target_keys.append(f"np_cm_{prefix}_leg_{suffix}")
        elif is_epic:
            target_keys.append(f"np_cm_{prefix}_{suffix}")
            target_keys.append(f"np_cm_{prefix}_epic")
            
        # 1. Сначала ищем по точному системному совпадению ключа (с исправленным багом регистра)
        for key in target_keys:
            # Убираем np_ из обоих ключей для 100% безопасного посимвольного сравнения!
            clean_target = key.lower().replace("np_", "").replace("_", "")
            for np_key, np_val in name_parts.items():
                clean_np_key = np_key.lower().replace("np_", "").replace("_", "")
                if clean_np_key == clean_target:
                    return np_val.get("fields", {}).get("PartName", "Unknown")
                    
        # 2. Резервный поиск по содержанию префикса класса и суффикса (если разработчики опечатались)
        for np_key, np_val in name_parts.items():
            k_lower = np_key.lower()
            if prefix in k_lower and suffix in k_lower:
                return np_val.get("fields", {}).get("PartName", "Unknown")
                
    return "Unknown"

# Применяем исправленную двухфакторную функцию к нашему DataFrame
df["Display_Name"] = df.apply(lambda row: resolve_display_name(row["Item_Code"], row["Internal_Category"], name_parts_data), axis=1)

print("Названия классмодов успешно расшифрованы!")
df[["Item_Code", "Character", "Display_Name"]].head()

Названия классмодов успешно расшифрованы!


,Item_Code,Character,Display_Name
0,classmod_dark_siren.comp_05_legendary_06,Vex (Siren),Teen Witch
1,classmod_dark_siren.comp_05_legendary_05,Vex (Siren),Illusionist
2,classmod_dark_siren.comp_05_legendary_04,Vex (Siren),Kindread Spirits
3,classmod_dark_siren.comp_05_legendary_03,Vex (Siren),Undead Eye
4,classmod_dark_siren.comp_05_legendary_02,Vex (Siren),Avatar


Ячейка 6: Привязка источников выпадения (Боссы)

In [6]:
with open(RAW_DATA_DIR / "NexusConfigStoreItemPoolList.json", "r", encoding="utf-8") as f:
    item_pool_list_data = json.load(f)

drop_sources = {}

def clean_handle(handle):
    if not handle or not isinstance(handle, str):
        return ""
    return handle.lower().replace("inv'", "").replace("'", "").strip()

# Сканируем всю базу пулов добычи боссов
for list_key, list_val in item_pool_list_data.items():
    boss_name = list_key.replace("ItemPoolList_", "").replace("_", " ").title()
    item_pools = list_val.get("fields", {}).get("ItemPools", [])
    
    for pool_entry in item_pools:
        itempool = pool_entry.get("itempool", {})
        item_data = itempool.get("item", {})
        
        if item_data.get("bInstance") and "Instance" in item_data:
            instance = item_data["Instance"] or {}
            items_in_pool = instance.get("items", [])
            for pool_item in items_in_pool:
                inner_item = pool_item.get("item", {}).get("item", {})
                handle = inner_item.get("Handle")
                if handle:
                    cleaned_h = clean_handle(handle)
                    if cleaned_h:
                        drop_sources.setdefault(cleaned_h, []).append(boss_name)
        else:
            handle = item_data.get("Handle")
            if handle:
                cleaned_h = clean_handle(handle)
                if cleaned_h:
                    drop_sources.setdefault(cleaned_h, []).append(boss_name)

def get_drop_source(row):
    cleaned_code = clean_handle(row["Item_Code"])
    if not cleaned_code:
        return "-"
    sources = drop_sources.get(cleaned_code, [])
    return ", ".join(set(sources)) if sources else "-"

df["Drop_Source"] = df.apply(get_drop_source, axis=1)
print("Источники добычи классмодов успешно привязаны!")

Источники добычи классмодов успешно привязаны!


Ячейка 7: Модульный разбор, дешифратор умений по координатам и наследование

In [7]:
# Загружаем системные базы дерева умений, тултипов и стратегий имен
with open(RAW_DATA_DIR / "NexusConfigStoreUISkillTree.json", "r", encoding="utf-8") as f:
    ui_skill_tree_data = json.load(f)
with open(RAW_DATA_DIR / "NexusConfigStoreUIToolTipDataDefs.json", "r", encoding="utf-8") as f:
    tooltips_data = json.load(f)
with open(RAW_DATA_DIR / "NexusConfigStoreInventoryNameStrategy.json", "r", encoding="utf-8") as f:
    name_strategy_data = json.load(f)

# Карта сопоставления папок классмодов к их деревьям в UISkillTree
char_to_tree_key = {
    "classmod_dark_siren": "dark_siren_skill_trees",
    "classmod_paladin": "paladin_skill_trees",
    "classmod_exo_soldier": "exo_skill_trees",
    "classmod_gravitar": "gravitar_skill_trees",
    "classmod_robodealer": "robodealer_skill_trees"
}

# --- Вспомогательный транслятор системных координат в названия навыков ---
def translate_skill_coordinate(category, progress_graph, node_name, ui_skill_tree_data, tooltips_data):
    cat_clean = category.split(" | ")[-1].strip().lower() if " | " in category else category.strip().lower()
    tree_key = char_to_tree_key.get(cat_clean)
    if not tree_key: return None
        
    trees_def = ui_skill_tree_data.get(tree_key, {})
    skill_trees = trees_def.get("fields", {}).get("SkillTrees", [])
    
    graph_clean = re.search(r"'(.*?)'", progress_graph).group(1) if "'" in progress_graph else progress_graph
    graph_clean = graph_clean.strip().lower()
    node_clean = node_name.strip().lower()
    
    # 1. Поиск по точному совпадению ProgressGraph ветки
    for tree in skill_trees:
        for seg in tree.get("Segments", []):
            seg_graph = seg.get("ProgressGraph", "")
            seg_graph_clean = re.search(r"'(.*?)'", seg_graph).group(1) if "'" in seg_graph else seg_graph
            if seg_graph_clean.strip().lower() == graph_clean:
                for tier in seg.get("Tiers", []):
                    for node in tier.get("nodes", []):
                        if node.get("Name", "").strip().lower() == node_clean:
                            tooltip_ref = node.get("ToolTip", "")
                            tooltip_key = re.search(r"'(.*?)'", tooltip_ref).group(1) if "'" in tooltip_ref else tooltip_ref
                            tooltip_obj = tooltips_data.get(tooltip_key.strip(), {})
                            header = tooltip_obj.get("fields", {}).get("Header", "")
                            if header and isinstance(header, str):
                                return header
    return None

# --- ИСПРАВЛЕННЫЙ вложенный транслятор деталей без привязки бренда для классмодов ---
def get_com_part_info(part_code, category, translation_map):
    code_lower = part_code.lower()
    cleaned = part_code.replace("part_", "")
    
    cat_lower = category.lower()
    
    # 1. Сначала ищем перевод строго локально в категории этого персонажа
    model_name = translation_map.get(cat_lower, {}).get(code_lower)
    if not model_name:
        model_name = translation_map.get(cat_lower, {}).get(cleaned.lower())
        
    # 2. Если локально не найдено — безопасно ищем в глобальном пуле "234 | ClassMod"
    if not model_name:
        global_key = None
        for k in translation_map.keys():
            if "234 | classmod" in k:
                global_key = k
                break
        if global_key:
            model_name = translation_map[global_key].get(code_lower) or translation_map[global_key].get(cleaned.lower())
            
    if model_name:
        return model_name
    else:
        # Резервный вариант при отсутствии перевода
        display_code = cleaned
        for suffix in ["_pld", "_exo", "_grav", "_robo", "_ds", "_siren", "_paladin", "_robodealer"]:
            display_code = re.sub(rf"{suffix}\b", "", display_code, flags=re.IGNORECASE)
            display_code = re.sub(rf"\\b{suffix}_", "", display_code, flags=re.IGNORECASE)
        return display_code.replace("_", " ").title()

def extract_parts_by_slot_advanced(selection_rules, slot_key, category, translation_map, category_parts_by_slot):
    parts_list = []
    slot_key_lower = slot_key.lower()
    
    rules_lower = {}
    if isinstance(selection_rules, dict):
        rules_lower = {k.lower(): v for k, v in selection_rules.items()}
    
    if slot_key_lower in rules_lower:
        slot_val = rules_lower[slot_key_lower]
        parts_in_slot = slot_val.get("parts", [])
        for p in parts_in_slot:
            part_code = p.get("part", "")
            if part_code:
                formatted_part = get_com_part_info(part_code, category, translation_map)
                parts_list.append(formatted_part)
    else:
        inherited_parts = category_parts_by_slot.get(category, {}).get(slot_key_lower, [])
        for part_code in inherited_parts:
            formatted_part = get_com_part_info(part_code, category, translation_map)
            parts_list.append(formatted_part)
            
    return ", ".join(sorted(list(set(parts_list)))) if parts_list else "-"

# --- ШАГ 1: Строим глобальный словарь деталей с ПУТЕВЫМ фильтром и ДЕДУКТИВНЫМ поиском слотов ---
local_parts = {}
all_scanned_categories = character_com_categories + ["234 | ClassMod"]

for category in all_scanned_categories:
    cat_val = inventory_data.get(category, {})
    if not isinstance(cat_val, dict): continue
    parts_dict = cat_val.get("parts", {})
    if not isinstance(parts_dict, dict): continue
    
    cat_prefix = category.split(" | ")[-1] if " | " in category else category
    local_parts[category] = {}
    
    for part_id, part_val in parts_dict.items():
        if isinstance(part_val, dict):
            part_name = part_val.get("name", "")
            part_path = part_val.get("path", "")
            
            # Строгая защита от утечек: деталь добавляется в категорию только если её путь начинается с префикса этой категории!
            if isinstance(part_path, str) and part_path.startswith(f"{cat_prefix}."):
                slot_name = part_val.get("dependency_slot", "").strip().lower()
                
                # Дедуктивно определяем слот при отсутствии системного свойства
                if not slot_name:
                    part_name_lower = part_name.lower()
                    if part_name_lower.startswith("passive_") or "passive_points" in part_name_lower:
                        slot_name = "passive_points"
                    elif "part_firmware_" in part_name_lower or "_firmware" in part_name_lower:
                        slot_name = "firmware"
                    elif "part_unique_" in part_name_lower or "_unique" in part_name_lower:
                        slot_name = "unique"
                    elif "body" in part_name_lower or "class_mod_body" in part_name_lower:
                        slot_name = "class_mod_body"
                    elif "stat1" in part_name_lower or "group1" in part_name_lower or "part_stat_" in part_name_lower:
                        slot_name = "stat_group1"
                    elif "stat2" in part_name_lower or "group2" in part_name_lower or "part_stat2_" in part_name_lower:
                        slot_name = "stat_group2"
                        
                if slot_name:
                    local_parts[category].setdefault(slot_name, []).append(part_name)

category_parts_by_slot = {}
for category, cat_slots in local_parts.items():
    category_parts_by_slot[category] = {}
    parent_categories = ["234 | ClassMod"] # Общий родитель
    
    all_slots = ["class_mod_body", "body", "firmware", "action_skill_mod", "active_augment", "passive_points", "stat_group1", "stat_group2"]
    for slot in all_slots:
        parts_list = []
        parts_list.extend(cat_slots.get(slot, []))
        if not parts_list:
            for parent_cat in parent_categories:
                parts_list.extend(local_parts.get(parent_cat, {}).get(slot, []))
        category_parts_by_slot[category][slot] = list(set(parts_list))

# --- ШАГ 2: Реализуем наследование правил генерации ---
def merge_com_inheritance_rules(row, inventory_data):
    selection_rules = row.get("Selection_Rules", {})
    category = row.get("Internal_Category", "")
    all_rules = {}
    
    category_parts = inventory_data.get(category, {}).get("parts", {})
    base_templates = [
        "base_comp_01_common", "comp_01_common",
        "base_comp_02_uncommon", "comp_02_uncommon",
        "base_comp_03_rare", "comp_03_rare",
        "base_comp_04_epic", "comp_04_epic",
        "base_comp_05_legendary", "comp_05_legendary",
        "base_comp_06_pearlescent", "comp_06_pearlescent", "comp_06_pearl"
    ]
    
    if isinstance(category_parts, dict):
        for part_key, part_val in category_parts.items():
            if isinstance(part_val, dict):
                part_name = part_val.get("name", "")
                if part_name in base_templates:
                    rules = part_val.get("fields", {}).get("PartTypeSelectionRules", {})
                    if isinstance(rules, dict):
                        all_rules.update(rules)
                        
    if isinstance(selection_rules, dict):
        all_rules.update(selection_rules)
    return all_rules

df["Merged_Rules"] = df.apply(lambda row: merge_com_inheritance_rules(row, inventory_data), axis=1)

# --- ШАГ 3: Заполнение слотов модулей ---
all_game_slots = ["class_mod_body", "body", "firmware", "action_skill_mod", "active_augment", "stat_group1", "stat_group2"]

for slot in all_game_slots:
    df[slot] = df.apply(lambda row: extract_parts_by_slot_advanced(\
        row["Merged_Rules"], \
        slot, \
        row["Internal_Category"],\
        part_translation_map,\
        category_parts_by_slot\
    ), axis=1)

# --- ШАГ 4: СУПЕРНАДЁЖНЫЙ ФИЛЬТРУЮЩИЙ ДЕШИФРАТОР УМЕНИЙ (passive_points) НА ОСНОВЕ СТРАТЕГИИ ИМЕН ---
def decrypt_passive_points_for_row(row, inventory_data, ui_skill_tree_data, tooltips_data, part_translation_map, name_strategy_data):
    merged_rules = row["Merged_Rules"]
    category = row["Internal_Category"]
    rules_lower = {k.lower(): v for k, v in merged_rules.items()} if isinstance(merged_rules, dict) else {}
    
    # 1. Достаем принудительно установленный корпус классмода (body)
    forced_body = None
    if "class_mod_body" in rules_lower:
        body_parts = rules_lower["class_mod_body"].get("parts", [])
        if body_parts:
            forced_body = body_parts[0].get("part")
            
    # 2. Считываем разрешенные пары (ProgressGraph + NodeName) из стратегии именования для этого корпуса
    cat_clean = category.split(" | ")[-1].strip().lower() if " | " in category else category.strip().lower()
    mapping = {
        "classmod_dark_siren": "ClassModNameStrat_DarkSiren",
        "classmod_paladin": "ClassModNameStrat_Paladin",
        "classmod_exo_soldier": "ClassModNameStrat_ExoSoldier",
        "classmod_gravitar": "ClassModNameStrat_Gravitar",
        "classmod_robodealer": "ClassModNameStrat_RoboDealer"
    }
    
    strat_key = mapping.get(cat_clean)
    allowed_coordinates = []
    
    if strat_key and forced_body:
        strat_obj = name_strategy_data.get(strat_key, {})
        body_comb_data = strat_obj.get("fields", {}).get("NamingStrategy", {}).get("BodyAndPassivesCombinationData", {})
        body_strat = body_comb_data.get(forced_body, {})
        passives_list = body_strat.get("Passives", [])
        for p_entry in passives_list:
            pas_node = p_entry.get("Passive", {})
            p_graph = pas_node.get("ProgressGraph", "")
            # Очищаем ProgressGraph от GbxProgressGraphDef' на обеих сторонах сопоставления
            p_graph_clean = re.search(r"'(.*?)'", p_graph).group(1) if "'" in p_graph else p_graph
            node_name = pas_node.get("NodeName", "")
            if p_graph_clean and node_name:
                allowed_coordinates.append((p_graph_clean.strip().lower(), node_name.strip().lower()))

    # 3. Извлекаем все доступные детали passive_points
    part_codes = []
    if "passive_points" in rules_lower and rules_lower["passive_points"].get("parts"):
        parts_list = rules_lower["passive_points"].get("parts", [])
        for p in parts_list:
            code = p.get("part")
            if code:
                part_codes.append(code)
    else:
        part_codes = category_parts_by_slot.get(category, {}).get("passive_points", [])
        
    resolved_skills = []
    cat_val = inventory_data.get(category, {})
    parts_dict = cat_val.get("parts", {})
    
    # 4. Проверяем каждую деталь на соответствие разрешенным координатам!
    for part_code in part_codes:
        part_val = {}
        if isinstance(parts_dict, dict):
            for p_id, p_entry in parts_dict.items():
                if isinstance(p_entry, dict) and p_entry.get("name") == part_code:
                    part_val = p_entry
                    break
        if not part_val: continue
        
        aspects = part_val.get("fields", {}).get("Aspects", [])
        for aspect in aspects:
            if isinstance(aspect, dict) and "ClassModPassivesAspect" in aspect.get("structtype", ""):
                passives = aspect.get("Passives", [])
                for pas in passives:
                    p_graph = pas.get("ProgressGraph", "").strip().lower()
                    p_graph_clean = re.search(r"'(.*?)'", p_graph).group(1) if "'" in p_graph else p_graph
                    node_name = pas.get("NodeName", "").strip().lower()
                    
                    is_allowed = True
                    if allowed_coordinates:
                        is_allowed = (p_graph_clean.strip().lower(), node_name.strip().lower()) in allowed_coordinates
                        
                    if is_allowed:
                        skill_name = translate_skill_coordinate(category, p_graph_clean, node_name, ui_skill_tree_data, tooltips_data)
                        if skill_name:
                            resolved_skills.append(skill_name)
                            
    if resolved_skills:
        return ", ".join(sorted(list(set(resolved_skills))))
    return "-"

df["passive_points"] = df.apply(lambda row: decrypt_passive_points_for_row(row, inventory_data, ui_skill_tree_data, tooltips_data, part_translation_map, name_strategy_data), axis=1)

# --- ШАГ 5: ВЫРАВНИВАНИЕ НАЗВАНИЙ ЭПИЧЕСКИХ КЛАССМОДОВ ---
def fix_epic_display_names(row):
    display_name = row["Display_Name"]
    if display_name == "Unknown" and row["Rarity"] == "Epic":
        return f"{row['Character']} Epic Class Mod"
    return display_name

df["Display_Name"] = df.apply(fix_epic_display_names, axis=1)

print("Все модули, навыки и пассивные характеристики успешно обработаны!")

Все модули, навыки и пассивные характеристики успешно обработаны!


Ячейка 8: Форматирование и экспорт в CSV

In [8]:
final_df = df.copy()

# Переименовываем базовые колонки
final_df = final_df.rename(columns={
    "Display_Name": "Name",
    "Drop_Source": "Drop Source",
    "World_Drop": "World Drop",
    "Item_Code": "Item Code"
})

# Сверхнадежная фильтрация: удаляем только абстрактные шаблоны из общей папки ClassMod
final_df = final_df[final_df["Internal_Category"] != "234 | ClassMod"]

# Порядок столбцов
all_game_slots = ["class_mod_body", "body", "firmware", "action_skill_mod", "active_augment", "passive_points", "stat_group1", "stat_group2"]

columns_order = [
    "Item Code", "Name", "Rarity", "Character", "World Drop", "Drop Source"
] + all_game_slots

final_df = final_df[columns_order]

# Сохраняем результат
final_df.to_csv(output_file, index=False, encoding="utf-8")

print(f"Экспорт классмодов завершен! Таблица сохранена в: {output_file}")
print(f"Размерность: {final_df.shape[0]} строк на {final_df.shape[1]} колонок.")

Экспорт классмодов завершен! Таблица сохранена в: ../data/output/classmods_result_20260728_170024.csv
Размерность: 55 строк на 14 колонок.
